# Notebook 8 — Customer Decision Engine

## Objective

This notebook integrates the previously developed customer and demand models into one operational decision framework.

The goal is not to create another standalone predictive model.

The goal is to determine:

> **What should the business do next for each customer?**

The decision engine combines signals from:

1. **Repeat Purchase Propensity**
2. **Reactivation Propensity**
3. **Revenue Leakage Risk**
4. **Future Customer Value**
5. **Next-Best Product Recommendation**
6. **Demand Forecasting**

The output is a customer-level action table containing:

- customer identifier;
- current lifecycle state;
- priority score;
- risk level;
- opportunity level;
- recommended action;
- recommended product or product set;
- reason for recommendation;
- urgency;
- and operational eligibility.

---

## Core Principle

Prediction and decision are not the same thing.

A customer can simultaneously have:

- high future value;
- high leakage risk;
- moderate repeat propensity;
- and a strong product recommendation.

The decision engine must resolve these competing signals into one prioritised action.

The system must therefore be:

- deterministic;
- interpretable;
- auditable;
- temporally safe;
- and operationally conservative.

---

## Decision Engine Philosophy

The engine will prioritise customers using four broad objectives:

### 1. Protect

Prevent likely revenue loss among currently valuable customers.

### 2. Retain

Encourage repeat purchase among active customers with weakening engagement.

### 3. Reactivate

Target previously active customers with meaningful return propensity.

### 4. Grow

Recommend products and deepen value among healthy, active customers.

---

## Important Limitations

This engine is not a causal uplift model.

A high predicted response probability does not prove that outreach will cause incremental behaviour.

Operational interventions should therefore be evaluated using controlled experiments wherever possible.

The decision engine also does not incorporate:

- marketing contact cost;
- channel-level response cost;
- inventory availability;
- gross margin;
- campaign fatigue;
- consent / contact restrictions;
- or promotion-specific economics

unless those fields are explicitly added later.

---

## Step 73 — Decision Engine Architecture

The decision engine is structured in five layers.

### Layer 1 — Customer State

Determine whether the customer is:

- Active
- At Risk
- Dormant
- Reactivation Candidate
- High-Value Growth Candidate

### Layer 2 — Model Signals

Combine outputs from the completed models:

- repeat probability;
- reactivation probability;
- leakage probability;
- leakage exposure;
- future customer value;
- next-best product signal.

### Layer 3 — Business Priority

Determine whether the customer should be prioritised for:

- protection;
- retention;
- reactivation;
- growth;
- or no immediate action.

### Layer 4 — Recommended Action

Assign one operational action.

Examples:

- Protect High-Value Customer
- Retention Outreach
- Reactivation Campaign
- Cross-Sell Recommendation
- Monitor Only

### Layer 5 — Decision Confidence

Separate:

- strong decisions;
- moderate decisions;
- weak / insufficient-signal decisions.

This prevents every customer from being forced into an action.

---

## STEP 74 — INPUT CONTRACT
Before we start scoring, we need a clean contract for what Notebook 8 expects from the previous notebooks.
### 74.1 — Model input specification

In [ ]:
# ================================================================
# 74.1 — LOAD PERSISTENT MODEL OUTPUTS
# ================================================================

from pathlib import Path

import numpy as np
import pandas as pd

MODEL_OUTPUT_DIR = Path(
    "model_outputs"
)


MODEL_FILES = {
    "repeat":
        "repeat_purchase_scores.parquet",

    "reactivation":
        "reactivation_scores.parquet",

    "leakage":
        "revenue_leakage_scores.parquet",

    "fcv":
        "future_customer_value_scores.parquet",

    "recommendation":
        "next_best_product_scores.parquet",
}


for model_name, filename in (
    MODEL_FILES.items()
):

    filepath = (
        MODEL_OUTPUT_DIR
        /
        filename
    )

    assert filepath.exists(), (
        f"Missing {model_name} output: "
        f"{filepath}"
    )


print(
    "✓ All five persistent model interfaces found."
)

In [ ]:
# ================================================================
# 74.2 — LOAD MODEL INTERFACES
# ================================================================

repeat_scores = pd.read_parquet(
    MODEL_OUTPUT_DIR
    /
    MODEL_FILES[
        "repeat"
    ]
)


reactivation_scores = pd.read_parquet(
    MODEL_OUTPUT_DIR
    /
    MODEL_FILES[
        "reactivation"
    ]
)


leakage_scores = pd.read_parquet(
    MODEL_OUTPUT_DIR
    /
    MODEL_FILES[
        "leakage"
    ]
)


fcv_scores = pd.read_parquet(
    MODEL_OUTPUT_DIR
    /
    MODEL_FILES[
        "fcv"
    ]
)


recommendation_scores = pd.read_parquet(
    MODEL_OUTPUT_DIR
    /
    MODEL_FILES[
        "recommendation"
    ]
)


print(
    "✓ Model outputs loaded successfully."
)

In [ ]:
# ================================================================
# 74.3 — MODEL INTERFACE AUDIT
# ================================================================

model_interface_audit = pd.DataFrame(
    [
        {
            "Model":
                "Repeat Purchase",

            "Rows":
                len(
                    repeat_scores
                ),

            "Customers":
                repeat_scores[
                    "customer"
                ].nunique(),

            "First Score Date":
                repeat_scores[
                    "repeat_score_date"
                ].min(),

            "Last Score Date":
                repeat_scores[
                    "repeat_score_date"
                ].max(),
        },

        {
            "Model":
                "Reactivation",

            "Rows":
                len(
                    reactivation_scores
                ),

            "Customers":
                reactivation_scores[
                    "customer"
                ].nunique(),

            "First Score Date":
                reactivation_scores[
                    "prediction_date"
                ].min(),

            "Last Score Date":
                reactivation_scores[
                    "prediction_date"
                ].max(),
        },

        {
            "Model":
                "Revenue Leakage",

            "Rows":
                len(
                    leakage_scores
                ),

            "Customers":
                leakage_scores[
                    "customer"
                ].nunique(),

            "First Score Date":
                leakage_scores[
                    "prediction_date"
                ].min(),

            "Last Score Date":
                leakage_scores[
                    "prediction_date"
                ].max(),
        },

        {
            "Model":
                "Future Customer Value",

            "Rows":
                len(
                    fcv_scores
                ),

            "Customers":
                fcv_scores[
                    "customer"
                ].nunique(),

            "First Score Date":
                fcv_scores[
                    "prediction_date"
                ].min(),

            "Last Score Date":
                fcv_scores[
                    "prediction_date"
                ].max(),
        },

        {
            "Model":
                "Next Best Product",

            "Rows":
                len(
                    recommendation_scores
                ),

            "Customers":
                recommendation_scores[
                    "customer"
                ].nunique(),

            "First Score Date":
                recommendation_scores[
                    "prediction_date"
                ].min(),

            "Last Score Date":
                recommendation_scores[
                    "prediction_date"
                ].max(),
        },
    ]
)


display(
    model_interface_audit
)

In [ ]:
# ================================================================
# 74.4 — TEMPORAL PROVENANCE AUDIT
# ================================================================

temporal_provenance_audit = pd.DataFrame(
    {
        "Check": [
            "FCV latest development outcome <= score date",
            "Leakage probability bounded 0–1",
            "Reactivation probability bounded 0–1",
            "Repeat probability bounded 0–1",
            "FCV future-active probability bounded 0–1",
        ],

        "Pass": [
            (
                fcv_scores[
                    "latest_development_outcome_end"
                ]
                <=
                fcv_scores[
                    "prediction_date"
                ]
            ).all(),

            leakage_scores[
                "leakage_probability"
            ]
            .between(
                0,
                1
            )
            .all(),

            reactivation_scores[
                "reactivation_probability"
            ]
            .between(
                0,
                1
            )
            .all(),

            repeat_scores[
                "repeat_probability"
            ]
            .between(
                0,
                1
            )
            .all(),

            fcv_scores[
                "future_active_probability"
            ]
            .between(
                0,
                1
            )
            .all(),
        ],
    }
)


display(
    temporal_provenance_audit
)


assert (
    temporal_provenance_audit[
        "Pass"
    ]
    .all()
)

## Temporal Integration Rule

The model outputs cannot be merged solely on customer ID.

They represent different decision moments and different eligible populations.

Notebook 8 will therefore construct a **decision snapshot** first and then attach only model signals that were valid at or before that snapshot.

In particular:

- Repeat Purchase is an early-lifecycle score anchored to the first observed order.
- Reactivation is valid only for eligible inactivity episodes.
- Revenue Leakage and FCV are month-end customer snapshots.
- Next-Best Product is generated immediately after an observed customer order.

This prevents historical scores from being incorrectly interpreted as simultaneous current-state predictions.

## Step 75 — Common Decision Snapshot and Temporal Alignment

The upstream models do not produce predictions at identical moments.

A common decision snapshot is therefore required before customer actions can be orchestrated.

### Common Decision Date

The primary decision snapshot is:

> **31 August 2011**

This date is selected because:

- Revenue Leakage has an August month-end scoring population;
- Future Customer Value has an August month-end scoring population;
- Reactivation contains eligible inactivity episodes through 31 August;
- Next-Best-Product contains post-order recommendations through 31 August.

### Repeat Purchase Qualification

Repeat Purchase scores were generated after customers' first observed orders and the final available score date is February 2011.

They are therefore not interpreted as current August repeat probabilities.

Repeat Purchase will remain available as an **early-customer lifecycle signal**, but it will not be forced into the August decision snapshot as though it were contemporaneous.

### Temporal Principle

A model signal can be used only when:

1. it was generated on or before the decision date;
2. it remains relevant to the customer's state at the decision date;
3. no later customer event has invalidated the signal.

This is particularly important for:

- reactivation episodes; and
- post-order product recommendations.

In [ ]:
# ================================================================
# 75.2 — COMMON DECISION DATE
# ================================================================

DECISION_DATE = pd.Timestamp(
    "2011-08-31 23:59:59"
)

DECISION_PERIOD = pd.Period(
    "2011-08",
    freq="M"
)


print(
    "Decision date:",
    DECISION_DATE
)

print(
    "Decision period:",
    DECISION_PERIOD
)

In [ ]:
# ================================================================
# 75.3 — LOAD CANONICAL CLEANED RETAIL DATA
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np


CLEANED_DATA_PATH = Path(
    "cleaned_online_retail.parquet"
)


assert CLEANED_DATA_PATH.exists(), (
    f"Canonical cleaned dataset not found: "
    f"{CLEANED_DATA_PATH.resolve()}"
)


retail = pd.read_parquet(
    CLEANED_DATA_PATH
)


print(
    "✓ Canonical cleaned retail dataset loaded."
)

print(
    f"Path: {CLEANED_DATA_PATH.resolve()}"
)

print(
    f"Rows: {len(retail):,}"
)

print(
    f"Columns: {len(retail.columns):,}"
)


display(
    retail.head()
)

In [ ]:
# ================================================================
# 75.3B — CANONICAL DATA SCHEMA AUDIT
# ================================================================

print(
    "Canonical columns:"
)

for column in retail.columns:

    print(
        " -",
        column
    )


print(
    "\nData types:"
)

display(
    retail.dtypes
    .rename(
        "dtype"
    )
    .to_frame()
)

In [ ]:
# ================================================================
# 75.3C — REQUIRED FIELD AUDIT
# ================================================================

required_core_fields = [
    "date",
    "customer",
    "invoice",
]


preferred_fields = [
    "product",
    "region",
    "quantity",
    "unit_price",
    "revenue",
    "net_merchandise_revenue",
    "gross_merchandise_revenue",
    "non_merchandise_revenue",
    "is_cancellation",
    "is_return",
]


field_audit = pd.DataFrame(
    {
        "Field":
            (
                required_core_fields
                +
                preferred_fields
            ),

        "Required":
            (
                [True]
                *
                len(
                    required_core_fields
                )
                +
                [False]
                *
                len(
                    preferred_fields
                )
            ),
    }
)


field_audit[
    "Present"
] = (
    field_audit[
        "Field"
    ]
    .isin(
        retail.columns
    )
)


display(
    field_audit
)


missing_required = (
    field_audit.loc[
        field_audit[
            "Required"
        ]
        &
        ~field_audit[
            "Present"
        ],
        "Field"
    ]
    .tolist()
)


assert (
    len(
        missing_required
    )
    ==
    0
), (
    f"Missing required canonical fields: "
    f"{missing_required}"
)


print(
    "✓ Required Notebook 8 fields are present."
)

In [ ]:
# ================================================================
# 75.4 — STANDARDISE TRANSACTION FIELDS
# ================================================================

assert "date" in retail.columns
assert "customer" in retail.columns
assert "invoice" in retail.columns


retail[
    "date"
] = pd.to_datetime(
    retail[
        "date"
    ]
)


retail[
    "customer"
] = (
    pd.to_numeric(
        retail[
            "customer"
        ],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)


print(
    "✓ Core transaction fields standardised."
)

print(
    "Transaction date range:",
    retail["date"].min(),
    "to",
    retail["date"].max()
)

In [ ]:
# ================================================================
# 75.5 — QUALIFYING POSITIVE MERCHANDISE TRANSACTIONS
# CANONICAL CLEANED DATA VERSION
# ================================================================

required_merchandise_fields = [
    "date",
    "customer",
    "invoice",
    "quantity",
    "unit_price",
    "net_merchandise_revenue",
    "is_cancellation",
    "is_return",
]


missing_merchandise_fields = [
    column
    for column in required_merchandise_fields
    if column not in retail.columns
]


assert (
    len(missing_merchandise_fields)
    ==
    0
), (
    f"Missing canonical merchandise fields: "
    f"{missing_merchandise_fields}"
)


# ------------------------------------------------
# Canonical qualifying positive merchandise logic
# ------------------------------------------------

positive_merchandise_mask = (
    (
        retail[
            "net_merchandise_revenue"
        ]
        >
        0
    )
    &
    (
        retail[
            "quantity"
        ]
        >
        0
    )
    &
    (
        retail[
            "unit_price"
        ]
        >
        0
    )
    &
    (
        ~retail[
            "is_cancellation"
        ]
        .fillna(False)
    )
    &
    (
        ~retail[
            "is_return"
        ]
        .fillna(False)
    )
)


snapshot_revenue_column = (
    "net_merchandise_revenue"
)


snapshot_transactions = (
    retail.loc[
        positive_merchandise_mask
        &
        retail[
            "customer"
        ]
        .notna()
        &
        (
            retail[
                "date"
            ]
            <=
            DECISION_DATE
        )
    ]
    .copy()
)


print(
    "✓ Qualifying positive merchandise history created."
)

print(
    f"Rows: "
    f"{len(snapshot_transactions):,}"
)

print(
    f"Identified customers: "
    f"{snapshot_transactions['customer'].nunique():,}"
)

print(
    f"Revenue field: "
    f"{snapshot_revenue_column}"
)

print(
    f"Transaction range: "
    f"{snapshot_transactions['date'].min()} "
    f"to "
    f"{snapshot_transactions['date'].max()}"
)

In [ ]:
# ================================================================
# 75.5B — QUALIFYING TRANSACTION SANITY CHECK
# ================================================================

transaction_sanity_audit = pd.DataFrame(
    {
        "Check": [
            "All revenue > 0",
            "All quantity > 0",
            "All unit price > 0",
            "No cancellations",
            "No returns",
            "No missing customer IDs",
            "No transactions after decision date",
        ],

        "Pass": [
            (
                snapshot_transactions[
                    "net_merchandise_revenue"
                ]
                >
                0
            ).all(),

            (
                snapshot_transactions[
                    "quantity"
                ]
                >
                0
            ).all(),

            (
                snapshot_transactions[
                    "unit_price"
                ]
                >
                0
            ).all(),

            (
                ~snapshot_transactions[
                    "is_cancellation"
                ]
                .fillna(False)
            ).all(),

            (
                ~snapshot_transactions[
                    "is_return"
                ]
                .fillna(False)
            ).all(),

            snapshot_transactions[
                "customer"
            ]
            .notna()
            .all(),

            (
                snapshot_transactions[
                    "date"
                ]
                <=
                DECISION_DATE
            ).all(),
        ],
    }
)


display(
    transaction_sanity_audit
)


assert (
    transaction_sanity_audit[
        "Pass"
    ]
    .all()
)


print(
    "✓ Snapshot transaction population passed all checks."
)

## STEP 75.6 — Customer activity snapshot

In [ ]:
# ================================================================
# 75.6A — CUSTOMER WINDOW AGGREGATION HELPER
# ================================================================

def aggregate_customer_window(
    transactions,
    days,
    prefix
):

    start_date = (
        DECISION_DATE
        -
        pd.Timedelta(
            days=days
        )
    )


    window = (
        transactions.loc[
            transactions[
                "date"
            ]
            >
            start_date
        ]
        .copy()
    )


    result = (
        window
        .groupby(
            "customer",
            as_index=False
        )
        .agg(
            **{
                f"orders_prior_{prefix}":
                    (
                        "invoice",
                        "nunique"
                    ),

                f"revenue_prior_{prefix}":
                    (
                        snapshot_revenue_column,
                        "sum"
                    ),
            }
        )
    )


    return result

In [ ]:
# ================================================================
# 75.6B — BASE CUSTOMER SNAPSHOT
# ================================================================

customer_snapshot = (
    snapshot_transactions
    .groupby(
        "customer",
        as_index=False
    )
    .agg(
        first_order_date=(
            "date",
            "min"
        ),

        last_order_date=(
            "date",
            "max"
        ),

        lifetime_orders=(
            "invoice",
            "nunique"
        ),

        lifetime_revenue=(
            snapshot_revenue_column,
            "sum"
        ),
    )
)


customer_snapshot[
    "days_since_last_order"
] = (
    (
        DECISION_DATE
        -
        customer_snapshot[
            "last_order_date"
        ]
    )
    .dt.total_seconds()
    /
    86400
)


customer_snapshot[
    "customer_tenure_days"
] = (
    (
        DECISION_DATE
        -
        customer_snapshot[
            "first_order_date"
        ]
    )
    .dt.total_seconds()
    /
    86400
)


for (
    days,
    prefix
) in [
    (
        30,
        "30d"
    ),
    (
        90,
        "90d"
    ),
    (
        180,
        "180d"
    ),
]:

    window_summary = (
        aggregate_customer_window(
            snapshot_transactions,
            days=days,
            prefix=prefix
        )
    )


    customer_snapshot = (
        customer_snapshot
        .merge(
            window_summary,
            on="customer",
            how="left"
        )
    )


activity_columns = [
    "orders_prior_30d",
    "orders_prior_90d",
    "orders_prior_180d",
    "revenue_prior_30d",
    "revenue_prior_90d",
    "revenue_prior_180d",
]


customer_snapshot[
    activity_columns
] = (
    customer_snapshot[
        activity_columns
    ]
    .fillna(0)
)


print(
    f"Customer snapshot rows: "
    f"{len(customer_snapshot):,}"
)


display(
    customer_snapshot.head(
        10
    )
)

In [ ]:
# ================================================================
# 75.7 — CUSTOMER LIFECYCLE STATE
# ================================================================

def classify_customer_state(
    days_since_last_order
):

    if pd.isna(
        days_since_last_order
    ):

        return "Unclassified"


    if (
        days_since_last_order
        <=
        30
    ):

        return "Active"


    if (
        days_since_last_order
        <=
        90
    ):

        return "At Risk"


    if (
        days_since_last_order
        <=
        180
    ):

        return "Dormant"


    return "Long Dormant"


customer_snapshot[
    "customer_state"
] = (
    customer_snapshot[
        "days_since_last_order"
    ]
    .apply(
        classify_customer_state
    )
)


customer_state_summary = (
    customer_snapshot[
        "customer_state"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "customer_state"
    )
    .reset_index(
        name="customers"
    )
)


customer_state_summary[
    "customer_share"
] = (
    customer_state_summary[
        "customers"
    ]
    /
    customer_state_summary[
        "customers"
    ]
    .sum()
)


display(
    customer_state_summary
)

In [ ]:
# ================================================================
# 76.1 — AUGUST LEAKAGE SNAPSHOT
# ================================================================

leakage_august = (
    leakage_scores.loc[
        leakage_scores[
            "prediction_date"
        ]
        .dt.to_period(
            "M"
        )
        ==
        DECISION_PERIOD
    ]
    .copy()
)


assert (
    leakage_august[
        "customer"
    ]
    .is_unique
)


print(
    f"August leakage customers: "
    f"{len(leakage_august):,}"
)

In [ ]:
# ================================================================
# 76.2 — AUGUST FCV SNAPSHOT
# ================================================================

fcv_august = (
    fcv_scores.loc[
        fcv_scores[
            "prediction_date"
        ]
        .dt.to_period(
            "M"
        )
        ==
        DECISION_PERIOD
    ]
    .copy()
)


assert (
    fcv_august[
        "customer"
    ]
    .is_unique
)


print(
    f"August FCV customers: "
    f"{len(fcv_august):,}"
)

In [ ]:
# ================================================================
# 77.1 — VALID REACTIVATION SIGNALS
# ================================================================

reactivation_snapshot = (
    reactivation_scores.loc[
        reactivation_scores[
            "prediction_date"
        ]
        <=
        DECISION_DATE
    ]
    .copy()
)


reactivation_snapshot[
    "reactivation_score_age_days"
] = (
    (
        DECISION_DATE
        -
        reactivation_snapshot[
            "prediction_date"
        ]
    )
    .dt.total_seconds()
    /
    86400
)


reactivation_snapshot = (
    reactivation_snapshot.loc[
        reactivation_snapshot[
            "reactivation_score_age_days"
        ]
        .between(
            0,
            90
        )
    ]
    .copy()
)


reactivation_snapshot = (
    reactivation_snapshot
    .sort_values(
        [
            "customer",
            "prediction_date",
        ]
    )
    .drop_duplicates(
        subset=[
            "customer"
        ],
        keep="last"
    )
)


print(
    f"Customers with temporally valid "
    f"reactivation score: "
    f"{len(reactivation_snapshot):,}"
)

## STEP 78 — Temporally align Next-Best-Product

In [ ]:
# ================================================================
# 78.1 — LATEST RECOMMENDATION EPISODE
# ================================================================

latest_recommendation = (
    recommendation_scores.loc[
        recommendation_scores[
            "prediction_date"
        ]
        <=
        DECISION_DATE
    ]
    .sort_values(
        [
            "customer",
            "prediction_date",
        ]
    )
    .drop_duplicates(
        subset=[
            "customer"
        ],
        keep="last"
    )
    .copy()
)


print(
    f"Customers with historical recommendation: "
    f"{len(latest_recommendation):,}"
)

In [ ]:
# ================================================================
# 78.2 — CURRENT RECOMMENDATION VALIDITY
# ================================================================

recommendation_validity = (
    latest_recommendation[
        [
            "customer",
            "prediction_date",
            "invoice",
            "repeat_top1_product",
            "discovery_top1_product",
            "repeat_recommendations",
            "discovery_recommendations",
            "history_depth_band",
            "repeat_channel_status",
            "discovery_channel_status",
        ]
    ]
    .merge(
        customer_snapshot[
            [
                "customer",
                "last_order_date",
            ]
        ],
        on="customer",
        how="left"
    )
)


recommendation_validity[
    "recommendation_is_current"
] = (
    (
        recommendation_validity[
            "prediction_date"
        ]
        -
        recommendation_validity[
            "last_order_date"
        ]
    )
    .abs()
    .dt.total_seconds()
    <=
    60
)


current_recommendations = (
    recommendation_validity.loc[
        recommendation_validity[
            "recommendation_is_current"
        ]
    ]
    .copy()
)


print(
    f"Current recommendation customers: "
    f"{len(current_recommendations):,}"
)

In [ ]:
# ================================================================
# 79.1 — REPEAT PURCHASE SIGNAL AGE
# ================================================================

repeat_snapshot_audit = (
    repeat_scores[
        [
            "customer",
            "repeat_score_date",
            "repeat_probability",
            "repeat_propensity_segment",
        ]
    ]
    .copy()
)


repeat_snapshot_audit[
    "repeat_score_age_days"
] = (
    (
        DECISION_DATE
        -
        repeat_snapshot_audit[
            "repeat_score_date"
        ]
    )
    .dt.total_seconds()
    /
    86400
)


repeat_snapshot_audit[
    "repeat_signal_current"
] = (
    repeat_snapshot_audit[
        "repeat_score_age_days"
    ]
    .between(
        0,
        90
    )
)


repeat_signal_validity = pd.DataFrame(
    {
        "Metric": [
            "Repeat scores",
            "Current within 90 days",
            "Stale at decision snapshot",
        ],

        "Value": [
            len(
                repeat_snapshot_audit
            ),

            repeat_snapshot_audit[
                "repeat_signal_current"
            ]
            .sum(),

            (
                ~repeat_snapshot_audit[
                    "repeat_signal_current"
                ]
            )
            .sum(),
        ],
    }
)


display(
    repeat_signal_validity
)

## STEP 80 — Build common decision snapshot

In [ ]:
# ================================================================
# 80.1 — LEAKAGE FIELDS FOR DECISION ENGINE
# ================================================================

leakage_for_decision = (
    leakage_august[
        [
            "customer",
            "leakage_probability",
            "leakage_probability_percentile",
            "baseline_revenue",
            "leakage_exposure_score",
            "leakage_exposure_percentile",
            "economic_priority_band",
            "risk_band",
            "value_band",
        ]
    ]
    .copy()
)

In [ ]:
# ================================================================
# 80.2 — FCV FIELDS FOR DECISION ENGINE
# ================================================================

fcv_for_decision = (
    fcv_august[
        [
            "customer",
            "fcv_history_180d_revenue",
            "future_active_probability",
            "future_customer_value_90d",
            "fcv_percentile",
            "fcv_tier",
        ]
    ]
    .copy()
)

In [ ]:
# ================================================================
# 80.3 — REACTIVATION FIELDS FOR DECISION ENGINE
# ================================================================

reactivation_for_decision = (
    reactivation_snapshot[
        [
            "customer",
            "prediction_date",
            "reactivation_probability",
            "reactivation_score_age_days",
            "historical_order_count",
            "historical_revenue",
            "prior_reactivation_rate",
        ]
    ]
    .rename(
        columns={
            "prediction_date":
                "reactivation_prediction_date"
        }
    )
    .copy()
)

In [ ]:
# ================================================================
# 80.4 — RECOMMENDATION FIELDS FOR DECISION ENGINE
# ================================================================

recommendation_for_decision = (
    current_recommendations[
        [
            "customer",
            "prediction_date",
            "repeat_top1_product",
            "discovery_top1_product",
            "repeat_recommendations",
            "discovery_recommendations",
            "history_depth_band",
            "repeat_channel_status",
            "discovery_channel_status",
        ]
    ]
    .rename(
        columns={
            "prediction_date":
                "recommendation_prediction_date"
        }
    )
    .copy()
)

In [ ]:
# ================================================================
# 80.5 — COMMON DECISION SNAPSHOT
# ================================================================

decision_snapshot = (
    customer_snapshot
    .merge(
        leakage_for_decision,
        on="customer",
        how="left"
    )
    .merge(
        fcv_for_decision,
        on="customer",
        how="left"
    )
    .merge(
        reactivation_for_decision,
        on="customer",
        how="left"
    )
    .merge(
        recommendation_for_decision,
        on="customer",
        how="left"
    )
)


decision_snapshot[
    "decision_date"
] = (
    DECISION_DATE
)


print(
    f"Decision snapshot customers: "
    f"{len(decision_snapshot):,}"
)


assert (
    decision_snapshot[
        "customer"
    ]
    .is_unique
)


display(
    decision_snapshot.head(
        10
    )
)

## STEP 81 — State-based signal validity

In [ ]:
# ================================================================
# 81.1 — OPERATIONAL SIGNAL ELIGIBILITY
# ================================================================

decision_snapshot[
    "leakage_signal_available"
] = (
    decision_snapshot[
        "leakage_probability"
    ]
    .notna()
)


decision_snapshot[
    "fcv_signal_available"
] = (
    decision_snapshot[
        "future_customer_value_90d"
    ]
    .notna()
)


decision_snapshot[
    "reactivation_signal_available"
] = (
    decision_snapshot[
        "reactivation_probability"
    ]
    .notna()
)


decision_snapshot[
    "recommendation_signal_available"
] = (
    decision_snapshot[
        "repeat_top1_product"
    ]
    .notna()
    |
    decision_snapshot[
        "discovery_top1_product"
    ]
    .notna()
)


decision_snapshot[
    "reactivation_operationally_valid"
] = (
    decision_snapshot[
        "reactivation_signal_available"
    ]
    &
    decision_snapshot[
        "customer_state"
    ]
    .isin(
        [
            "Dormant",
            "Long Dormant",
        ]
    )
)


decision_snapshot[
    "leakage_operationally_valid"
] = (
    decision_snapshot[
        "leakage_signal_available"
    ]
    &
    decision_snapshot[
        "customer_state"
    ]
    .isin(
        [
            "Active",
            "At Risk",
        ]
    )
)


decision_snapshot[
    "recommendation_operationally_valid"
] = (
    decision_snapshot[
        "recommendation_signal_available"
    ]
    &
    decision_snapshot[
        "customer_state"
    ]
    .isin(
        [
            "Active",
            "At Risk",
        ]
    )
)

## STEP 82 — Coverage audit

In [ ]:
# ================================================================
# 82.1 — MODEL SIGNAL COVERAGE
# ================================================================

signal_coverage = pd.DataFrame(
    {
        "Signal": [
            "Leakage available",
            "Leakage operationally valid",
            "FCV available",
            "Reactivation available",
            "Reactivation operationally valid",
            "Recommendation available",
            "Recommendation operationally valid",
        ],

        "Customers": [
            decision_snapshot[
                "leakage_signal_available"
            ].sum(),

            decision_snapshot[
                "leakage_operationally_valid"
            ].sum(),

            decision_snapshot[
                "fcv_signal_available"
            ].sum(),

            decision_snapshot[
                "reactivation_signal_available"
            ].sum(),

            decision_snapshot[
                "reactivation_operationally_valid"
            ].sum(),

            decision_snapshot[
                "recommendation_signal_available"
            ].sum(),

            decision_snapshot[
                "recommendation_operationally_valid"
            ].sum(),
        ],
    }
)


signal_coverage[
    "Customer Share"
] = (
    signal_coverage[
        "Customers"
    ]
    /
    len(
        decision_snapshot
    )
)


display(
    signal_coverage
)

In [ ]:
# ================================================================
# 82.2 — SIGNAL COVERAGE BY CUSTOMER STATE
# ================================================================

signal_coverage_by_state = (
    decision_snapshot
    .groupby(
        "customer_state",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        leakage_available=(
            "leakage_signal_available",
            "sum"
        ),

        fcv_available=(
            "fcv_signal_available",
            "sum"
        ),

        reactivation_available=(
            "reactivation_signal_available",
            "sum"
        ),

        reactivation_valid=(
            "reactivation_operationally_valid",
            "sum"
        ),

        recommendation_available=(
            "recommendation_signal_available",
            "sum"
        ),

        recommendation_valid=(
            "recommendation_operationally_valid",
            "sum"
        ),
    )
)


display(
    signal_coverage_by_state
)

## STEP 83 — Cross-model overlap

In [ ]:
# ================================================================
# 83.1 — MODEL OVERLAP PATTERNS
# ================================================================

decision_snapshot[
    "signal_count"
] = (
    decision_snapshot[
        [
            "leakage_signal_available",
            "fcv_signal_available",
            "reactivation_signal_available",
            "recommendation_signal_available",
        ]
    ]
    .sum(
        axis=1
    )
)


signal_count_summary = (
    decision_snapshot[
        "signal_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "Number of Available Signals"
    )
    .reset_index(
        name="Customers"
    )
)


signal_count_summary[
    "Customer Share"
] = (
    signal_count_summary[
        "Customers"
    ]
    /
    signal_count_summary[
        "Customers"
    ]
    .sum()
)


display(
    signal_count_summary
)

In [ ]:
# ================================================================
# 83.2 — MODEL PAIR OVERLAP
# ================================================================

signal_columns = [
    "leakage_signal_available",
    "fcv_signal_available",
    "reactivation_signal_available",
    "recommendation_signal_available",
]


signal_overlap_matrix = (
    decision_snapshot[
        signal_columns
    ]
    .astype(int)
    .T
    .dot(
        decision_snapshot[
            signal_columns
        ]
        .astype(int)
    )
)


display(
    signal_overlap_matrix
)

In [ ]:
# ================================================================
# 84.1 — DECISION SNAPSHOT VALIDATION
# ================================================================

assert (
    decision_snapshot[
        "customer"
    ]
    .is_unique
)


assert (
    (
        decision_snapshot[
            "last_order_date"
        ]
        <=
        DECISION_DATE
    )
    .all()
)


assert (
    (
        decision_snapshot[
            "days_since_last_order"
        ]
        >=
        0
    )
    .all()
)


assert (
    decision_snapshot[
        "customer_state"
    ]
    .notna()
    .all()
)


print(
    "✓ Common decision snapshot created."
)

print(
    "✓ Customer lifecycle state calculated using only history available by snapshot."
)

print(
    "✓ Leakage and FCV aligned to August month-end."
)

print(
    "✓ Reactivation limited to temporally valid episodes."
)

print(
    "✓ Stale product recommendations invalidated."
)

print(
    "✓ Repeat Purchase retained outside the current-state orchestration."
)

## Step 85 — Customer Action Orchestration

The common decision snapshot supports a state-aware orchestration framework.

The model signals are not treated as interchangeable probabilities.

Instead, customer lifecycle state determines which model is relevant.

### Decision Hierarchy

#### Active Customers

Active customers may receive one of three actions:

1. **Protect** when current revenue exposure is materially at risk;
2. **Grow** when the customer is commercially healthy and a current product recommendation is available;
3. **Monitor** when neither intervention is strongly supported.

#### At-Risk Customers

At-Risk customers receive:

> **Retain**

Their recency state already indicates weakening engagement, while the Leakage model provides economic-risk prioritisation.

#### Dormant Customers

Dormant customers receive:

> **Reactivate**

only when a temporally valid Reactivation propensity score is available.

Dormant customers without a valid Reactivation score remain:

> **Monitor — insufficient action signal**

#### Long-Dormant Customers

Long-Dormant customers have no contemporaneous model coverage in the current framework.

They are therefore classified as:

> **Unscored — Long Dormant**

No model-supported intervention is inferred for this group.

### Governance Principle

The engine does not force every customer into a campaign.

Absence of a valid model signal is treated as information rather than replaced with an arbitrary rule.

In [ ]:
# ================================================================
# 85.2 — ACTIVE CUSTOMER PROTECTION ELIGIBILITY
# ================================================================

decision_snapshot[
    "high_leakage_exposure"
] = (
    decision_snapshot[
        "economic_priority_band"
    ]
    .astype("string")
    .str.startswith(
        (
            "P1",
            "P2",
        ),
        na=False
    )
)


decision_snapshot[
    "protect_eligible"
] = (
    (
        decision_snapshot[
            "customer_state"
        ]
        ==
        "Active"
    )
    &
    decision_snapshot[
        "leakage_operationally_valid"
    ]
    &
    decision_snapshot[
        "high_leakage_exposure"
    ]
)


print(
    "Active customers eligible for Protect:",
    decision_snapshot[
        "protect_eligible"
    ].sum()
)

In [ ]:
# ================================================================
# 85.3 — GROWTH ELIGIBILITY
# ================================================================

decision_snapshot[
    "grow_eligible"
] = (
    (
        decision_snapshot[
            "customer_state"
        ]
        ==
        "Active"
    )
    &
    (
        ~decision_snapshot[
            "protect_eligible"
        ]
    )
    &
    decision_snapshot[
        "recommendation_operationally_valid"
    ]
    &
    decision_snapshot[
        "fcv_signal_available"
    ]
)


print(
    "Active customers eligible for Grow:",
    decision_snapshot[
        "grow_eligible"
    ].sum()
)

In [ ]:
# ================================================================
# 85.4 — RETENTION ELIGIBILITY
# ================================================================

decision_snapshot[
    "retain_eligible"
] = (
    (
        decision_snapshot[
            "customer_state"
        ]
        ==
        "At Risk"
    )
    &
    decision_snapshot[
        "leakage_operationally_valid"
    ]
)


print(
    "At-Risk customers eligible for Retain:",
    decision_snapshot[
        "retain_eligible"
    ].sum()
)

In [ ]:
# ================================================================
# 85.5 — REACTIVATION ELIGIBILITY
# ================================================================

decision_snapshot[
    "reactivate_eligible"
] = (
    (
        decision_snapshot[
            "customer_state"
        ]
        ==
        "Dormant"
    )
    &
    decision_snapshot[
        "reactivation_operationally_valid"
    ]
)


print(
    "Dormant customers eligible for Reactivate:",
    decision_snapshot[
        "reactivate_eligible"
    ].sum()
)

## STEP 86 — ACTION-SPECIFIC PRIORITY SCORES
These are ranking scores, not predictive probabilities.
### 86.1 — Protect priority
Leakage exposure should dominate because Notebook 4 showed that exposure ranking captured actual economic decline much better than probability ranking.

In [ ]:
# ================================================================
# 86.1 — PROTECT PRIORITY SCORE
# ================================================================

decision_snapshot[
    "protect_priority_score"
] = (
    0.60
    *
    decision_snapshot[
        "leakage_exposure_percentile"
    ]
    +
    0.25
    *
    decision_snapshot[
        "leakage_probability"
    ]
    +
    0.15
    *
    decision_snapshot[
        "fcv_percentile"
    ]
)

In [ ]:
# ================================================================
# 86.2 — RETAIN PRIORITY SCORE
# ================================================================

decision_snapshot[
    "retain_priority_score"
] = (
    0.50
    *
    decision_snapshot[
        "leakage_exposure_percentile"
    ]
    +
    0.30
    *
    decision_snapshot[
        "leakage_probability"
    ]
    +
    0.20
    *
    decision_snapshot[
        "fcv_percentile"
    ]
)

In [ ]:
# ================================================================
# 86.3 — REACTIVATION PRIORITY SCORE
# ================================================================

decision_snapshot[
    "reactivate_priority_score"
] = (
    0.65
    *
    decision_snapshot[
        "reactivation_probability"
    ]
    +
    0.35
    *
    decision_snapshot[
        "fcv_percentile"
    ]
)

In [ ]:
# ================================================================
# 86.4 — GROWTH PRIORITY SCORE
# ================================================================

decision_snapshot[
    "grow_priority_score"
] = (
    0.55
    *
    decision_snapshot[
        "fcv_percentile"
    ]
    +
    0.45
    *
    decision_snapshot[
        "future_active_probability"
    ]
)

### Priority Score Interpretation

The Protect, Retain, Reactivate and Grow priority scores are operational ranking scores.

They are **not probabilities**.

Their weights encode business policy:

- Protect prioritises economic exposure;
- Retain balances exposure, leakage risk and customer value;
- Reactivate prioritises return propensity with customer value;
- Grow prioritises future customer value and expected continued activity.

These weights have not been causally optimised.

They should eventually be replaced or recalibrated using controlled intervention economics such as incremental response, margin and contact cost.

## STEP 87 — ASSIGN ONE PRIMARY ACTION

In [ ]:
# ================================================================
# 87.1 — PRIMARY ACTION ASSIGNMENT
# ================================================================

def assign_primary_action(
    row
):

    state = row[
        "customer_state"
    ]


    # ------------------------------------------------
    # ACTIVE
    # ------------------------------------------------

    if (
        state
        ==
        "Active"
    ):

        if row[
            "protect_eligible"
        ]:

            return "Protect"

        if row[
            "grow_eligible"
        ]:

            return "Grow"

        return "Monitor"


    # ------------------------------------------------
    # AT RISK
    # ------------------------------------------------

    if (
        state
        ==
        "At Risk"
    ):

        if row[
            "retain_eligible"
        ]:

            return "Retain"

        return "Monitor"


    # ------------------------------------------------
    # DORMANT
    # ------------------------------------------------

    if (
        state
        ==
        "Dormant"
    ):

        if row[
            "reactivate_eligible"
        ]:

            return "Reactivate"

        return "Monitor — No Reactivation Score"


    # ------------------------------------------------
    # LONG DORMANT
    # ------------------------------------------------

    if (
        state
        ==
        "Long Dormant"
    ):

        return "Unscored — Long Dormant"


    return "Unclassified"


decision_snapshot[
    "primary_action"
] = (
    decision_snapshot.apply(
        assign_primary_action,
        axis=1
    )
)

In [ ]:
# ================================================================
# 87.2 — PRIMARY ACTION PRIORITY SCORE
# ================================================================

def get_primary_priority_score(
    row
):

    action = row[
        "primary_action"
    ]


    if (
        action
        ==
        "Protect"
    ):

        return row[
            "protect_priority_score"
        ]


    if (
        action
        ==
        "Retain"
    ):

        return row[
            "retain_priority_score"
        ]


    if (
        action
        ==
        "Reactivate"
    ):

        return row[
            "reactivate_priority_score"
        ]


    if (
        action
        ==
        "Grow"
    ):

        return row[
            "grow_priority_score"
        ]


    return np.nan


decision_snapshot[
    "primary_action_score"
] = (
    decision_snapshot.apply(
        get_primary_priority_score,
        axis=1
    )
)

## STEP 88 — PRIORITY TIERS
Rank customers within their assigned action, not against customers solving different business problems.

In [ ]:
# ================================================================
# 88.1 — ACTION-SPECIFIC PRIORITY PERCENTILE
# ================================================================

actionable_actions = [
    "Protect",
    "Retain",
    "Reactivate",
    "Grow",
]


actionable_mask = (
    decision_snapshot[
        "primary_action"
    ]
    .isin(
        actionable_actions
    )
)


decision_snapshot[
    "action_priority_percentile"
] = np.nan


decision_snapshot.loc[
    actionable_mask,
    "action_priority_percentile"
] = (
    decision_snapshot.loc[
        actionable_mask
    ]
    .groupby(
        "primary_action"
    )[
        "primary_action_score"
    ]
    .rank(
        pct=True,
        method="average"
    )
)

In [ ]:
# ================================================================
# 88.2 — PRIORITY TIER
# ================================================================

def assign_priority_tier(
    row
):

    if (
        row[
            "primary_action"
        ]
        not in
        actionable_actions
    ):

        return "Not Prioritised"


    percentile = row[
        "action_priority_percentile"
    ]


    if (
        percentile
        >=
        0.90
    ):

        return "Priority 1"


    if (
        percentile
        >=
        0.70
    ):

        return "Priority 2"


    if (
        percentile
        >=
        0.40
    ):

        return "Priority 3"


    return "Priority 4"


decision_snapshot[
    "priority_tier"
] = (
    decision_snapshot.apply(
        assign_priority_tier,
        axis=1
    )
)

## STEP 89 — PRODUCT RECOMMENDATION ORCHESTRATION
The same product channel should not be shown for every action.

In [ ]:
# ================================================================
# 89.1 — RECOMMENDATION CHANNEL BY ACTION
# ================================================================

def assign_recommendation_channel(
    row
):

    action = row[
        "primary_action"
    ]


    if not row[
        "recommendation_operationally_valid"
    ]:

        return pd.NA


    if (
        action
        ==
        "Grow"
    ):

        return "Discovery"


    if (
        action
        in [
            "Protect",
            "Retain",
        ]
    ):

        return "Repeat"


    return pd.NA


decision_snapshot[
    "recommended_product_channel"
] = (
    decision_snapshot.apply(
        assign_recommendation_channel,
        axis=1
    )
)

In [ ]:
# ================================================================
# 89.2 — FINAL TOP-1 PRODUCT
# ================================================================

def select_recommended_product(
    row
):

    channel = row[
        "recommended_product_channel"
    ]


    if (
        channel
        ==
        "Repeat"
    ):

        return row[
            "repeat_top1_product"
        ]


    if (
        channel
        ==
        "Discovery"
    ):

        return row[
            "discovery_top1_product"
        ]


    return pd.NA


decision_snapshot[
    "recommended_product"
] = (
    decision_snapshot.apply(
        select_recommended_product,
        axis=1
    )
)

In [ ]:
# ================================================================
# 90.1 — DECISION REASON
# ================================================================

def build_decision_reason(
    row
):

    action = row[
        "primary_action"
    ]


    if (
        action
        ==
        "Protect"
    ):

        return (
            "Active customer with high economic "
            "revenue-leakage exposure"
        )


    if (
        action
        ==
        "Retain"
    ):

        return (
            "Customer is 31–90 days since last purchase "
            "and has a current revenue-leakage score"
        )


    if (
        action
        ==
        "Reactivate"
    ):

        return (
            "Dormant customer with a temporally valid "
            "90-day reactivation propensity score"
        )


    if (
        action
        ==
        "Grow"
    ):

        return (
            "Active customer with current product recommendation "
            "and future-value signal"
        )


    if (
        action
        ==
        "Monitor — No Reactivation Score"
    ):

        return (
            "Dormant customer but no valid Reactivation "
            "score exists at the decision snapshot"
        )


    if (
        action
        ==
        "Unscored — Long Dormant"
    ):

        return (
            "Customer exceeds 180 days since last purchase "
            "and lies outside current model coverage"
        )


    return (
        "No model-supported intervention currently "
        "takes precedence"
    )


decision_snapshot[
    "decision_reason"
] = (
    decision_snapshot.apply(
        build_decision_reason,
        axis=1
    )
)

## STEP 91 — INITIAL ACTION AUDIT

In [ ]:
# ================================================================
# 91.1 — PRIMARY ACTION DISTRIBUTION
# ================================================================

action_distribution = (
    decision_snapshot[
        "primary_action"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "primary_action"
    )
    .reset_index(
        name="customers"
    )
)


action_distribution[
    "customer_share"
] = (
    action_distribution[
        "customers"
    ]
    /
    len(
        decision_snapshot
    )
)


display(
    action_distribution
)

In [ ]:
# ================================================================
# 91.2 — ACTION / STATE MATRIX
# ================================================================

action_state_matrix = pd.crosstab(
    decision_snapshot[
        "customer_state"
    ],

    decision_snapshot[
        "primary_action"
    ],

    margins=True
)


display(
    action_state_matrix
)

In [ ]:
# ================================================================
# 91.3 — PRIORITY DISTRIBUTION BY ACTION
# ================================================================

priority_action_summary = (
    decision_snapshot.loc[
        decision_snapshot[
            "primary_action"
        ]
        .isin(
            actionable_actions
        )
    ]
    .groupby(
        [
            "primary_action",
            "priority_tier",
        ],
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        mean_priority_score=(
            "primary_action_score",
            "mean"
        ),

        mean_fcv_percentile=(
            "fcv_percentile",
            "mean"
        ),

        mean_leakage_probability=(
            "leakage_probability",
            "mean"
        ),

        mean_leakage_exposure=(
            "leakage_exposure_score",
            "mean"
        ),

        mean_reactivation_probability=(
            "reactivation_probability",
            "mean"
        ),
    )
)


display(
    priority_action_summary
)

In [ ]:
# ================================================================
# 91.4 — PRODUCT RECOMMENDATION COVERAGE BY ACTION
# ================================================================

action_recommendation_coverage = (
    decision_snapshot
    .groupby(
        "primary_action",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        customers_with_product=(
            "recommended_product",
            lambda values:
                values.notna().sum()
        ),
    )
)


action_recommendation_coverage[
    "product_recommendation_coverage"
] = (
    action_recommendation_coverage[
        "customers_with_product"
    ]
    /
    action_recommendation_coverage[
        "customers"
    ]
)


display(
    action_recommendation_coverage
)

In [ ]:
# ================================================================
# 92.1 — DECISION ENGINE LOGIC CHECKS
# ================================================================

decision_logic_checks = pd.DataFrame(
    {
        "Check": [
            "Protect only assigned to Active",
            "Retain only assigned to At Risk",
            "Reactivate only assigned to Dormant",
            "Grow only assigned to Active",
            "Long Dormant receives no model-led action",
            "Reactivate always has valid Reactivation signal",
            "Grow always has valid recommendation",
        ],

        "Pass": [
            (
                decision_snapshot.loc[
                    decision_snapshot[
                        "primary_action"
                    ]
                    ==
                    "Protect",
                    "customer_state"
                ]
                ==
                "Active"
            ).all(),

            (
                decision_snapshot.loc[
                    decision_snapshot[
                        "primary_action"
                    ]
                    ==
                    "Retain",
                    "customer_state"
                ]
                ==
                "At Risk"
            ).all(),

            (
                decision_snapshot.loc[
                    decision_snapshot[
                        "primary_action"
                    ]
                    ==
                    "Reactivate",
                    "customer_state"
                ]
                ==
                "Dormant"
            ).all(),

            (
                decision_snapshot.loc[
                    decision_snapshot[
                        "primary_action"
                    ]
                    ==
                    "Grow",
                    "customer_state"
                ]
                ==
                "Active"
            ).all(),

            (
                decision_snapshot.loc[
                    decision_snapshot[
                        "customer_state"
                    ]
                    ==
                    "Long Dormant",
                    "primary_action"
                ]
                ==
                "Unscored — Long Dormant"
            ).all(),

            decision_snapshot.loc[
                decision_snapshot[
                    "primary_action"
                ]
                ==
                "Reactivate",
                "reactivation_operationally_valid"
            ]
            .all(),

            decision_snapshot.loc[
                decision_snapshot[
                    "primary_action"
                ]
                ==
                "Grow",
                "recommendation_operationally_valid"
            ]
            .all(),
        ],
    }
)


display(
    decision_logic_checks
)


assert (
    decision_logic_checks[
        "Pass"
    ]
    .all()
)


print(
    "✓ Initial decision orchestration passed logic checks."
)

## STEP 93 — CONTACT POLICY
Rather than contacting every customer assigned an action, use the priority tiers operationally:
- Priority 1: immediate intervention
- Priority 2: active campaign queue
- Priority 3: lower-cost / automated intervention
- Priority 4: hold / control / monitor

In [ ]:
# ================================================================
# 93.1 — CONTACT INTENSITY POLICY
# ================================================================

def assign_contact_intensity(row):

    action = row["primary_action"]
    tier = row["priority_tier"]

    if action not in actionable_actions:
        return "No Model-Led Contact"

    if tier == "Priority 1":
        return "High Touch"

    if tier == "Priority 2":
        return "Campaign"

    if tier == "Priority 3":
        return "Automated / Low Cost"

    return "Hold / Control"


decision_snapshot[
    "contact_intensity"
] = (
    decision_snapshot.apply(
        assign_contact_intensity,
        axis=1
    )
)

## STEP 94 — BUSINESS ACTION PLAYBOOK
Now translate the model decision into something the commercial team can actually do.

In [ ]:
# ================================================================
# 94.1 — BUSINESS ACTION PLAYBOOK
# ================================================================

def assign_business_action(row):

    action = row["primary_action"]

    if action == "Protect":

        if pd.notna(row["recommended_product"]):
            return (
                "Protect customer value; review leakage risk and "
                "reinforce likely repeat purchase"
            )

        return (
            "Protect customer value; investigate leakage drivers "
            "and preserve current spend"
        )

    if action == "Retain":

        if pd.notna(row["recommended_product"]):
            return (
                "Retention outreach using relevant repeat-product cue"
            )

        return (
            "Retention outreach focused on re-engagement and "
            "relationship continuity"
        )

    if action == "Reactivate":

        return (
            "Controlled reactivation campaign; test outreach against "
            "appropriate holdout/control group"
        )

    if action == "Grow":

        return (
            "Cross-sell / discovery offer using current "
            "next-best-product recommendation"
        )

    if action == "Monitor — No Reactivation Score":

        return (
            "Monitor dormant customer; insufficient current model "
            "evidence for targeted intervention"
        )

    if action == "Unscored — Long Dormant":

        return (
            "Outside current model coverage; route to separate "
            "deep-reactivation strategy"
        )

    return "Monitor customer without immediate intervention"


decision_snapshot[
    "business_action"
] = (
    decision_snapshot.apply(
        assign_business_action,
        axis=1
    )
)

## STEP 95 — EXPERIMENTATION FLAG
Retain, Reactivate and Discovery/Grow interventions should explicitly be treated as experiments rather than assumed causal treatments.

In [ ]:
# ================================================================
# 95.1 — EXPERIMENTATION REQUIREMENT
# ================================================================

def assign_experiment_requirement(row):

    action = row["primary_action"]

    if action == "Reactivate":
        return "Required"

    if action == "Grow":
        return "Required — Discovery channel is CONDITIONAL GO"

    if action == "Retain":
        return "Recommended"

    if action == "Protect":
        return "Recommended"

    return "Not Applicable"


decision_snapshot[
    "experiment_requirement"
] = (
    decision_snapshot.apply(
        assign_experiment_requirement,
        axis=1
    )
)

## STEP 96 — CONTACT QUEUE AUDIT

In [ ]:
# ================================================================
# 96.1 — CONTACT INTENSITY SUMMARY
# ================================================================

contact_intensity_summary = (
    decision_snapshot
    .groupby(
        [
            "primary_action",
            "contact_intensity",
        ],
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        )
    )
)


display(
    contact_intensity_summary
)

In [ ]:
# ================================================================
# 96.2 — IMMEDIATE COMMERCIAL WORKLOAD
# ================================================================

immediate_workload = (
    decision_snapshot.loc[
        decision_snapshot[
            "priority_tier"
        ]
        .isin(
            [
                "Priority 1",
                "Priority 2",
            ]
        )
    ]
    .groupby(
        "primary_action",
        as_index=False
    )
    .agg(
        customers=(
            "customer",
            "nunique"
        ),

        priority_1=(
            "priority_tier",
            lambda x:
                (x == "Priority 1").sum()
        ),

        priority_2=(
            "priority_tier",
            lambda x:
                (x == "Priority 2").sum()
        ),
    )
)


immediate_workload[
    "share_of_immediate_queue"
] = (
    immediate_workload[
        "customers"
    ]
    /
    immediate_workload[
        "customers"
    ]
    .sum()
)


display(
    immediate_workload
)

## STEP 97 — CUSTOMER DECISION TABLE
This becomes the operational output of Notebook 8.

In [ ]:
# ================================================================
# 97.1 — FINAL CUSTOMER DECISION TABLE
# ================================================================

decision_output_columns = [
    "customer",
    "decision_date",

    # Lifecycle
    "customer_state",
    "days_since_last_order",
    "customer_tenure_days",
    "lifetime_orders",
    "lifetime_revenue",
    "orders_prior_30d",
    "orders_prior_90d",
    "revenue_prior_90d",
    "revenue_prior_180d",

    # Decision
    "primary_action",
    "primary_action_score",
    "action_priority_percentile",
    "priority_tier",
    "contact_intensity",
    "business_action",
    "decision_reason",
    "experiment_requirement",

    # Leakage
    "leakage_probability",
    "leakage_exposure_score",
    "leakage_exposure_percentile",
    "economic_priority_band",

    # FCV
    "future_active_probability",
    "future_customer_value_90d",
    "fcv_percentile",
    "fcv_tier",

    # Reactivation
    "reactivation_probability",
    "reactivation_prediction_date",

    # Product recommendation
    "recommended_product_channel",
    "recommended_product",
    "repeat_top1_product",
    "discovery_top1_product",
]


customer_decision_engine_output = (
    decision_snapshot[
        decision_output_columns
    ]
    .copy()
)


customer_decision_engine_output = (
    customer_decision_engine_output
    .sort_values(
        [
            "priority_tier",
            "primary_action",
            "primary_action_score",
        ],
        ascending=[
            True,
            True,
            False,
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    customer_decision_engine_output.head(
        20
    )
)

In [ ]:
# ================================================================
# 98.1 — FINAL ENGINE GOVERNANCE CHECKS
# ================================================================

final_engine_checks = pd.DataFrame(
    {
        "Check": [
            "One row per customer",
            "All customers have primary action",
            "All actionable customers have priority tier",
            "All Grow customers have product recommendation",
            "All Reactivate customers have reactivation probability",
            "No Long Dormant customer receives model-led action",
            "No stale Repeat Purchase score used",
        ],

        "Pass": [
            customer_decision_engine_output[
                "customer"
            ].is_unique,

            customer_decision_engine_output[
                "primary_action"
            ].notna().all(),

            customer_decision_engine_output.loc[
                customer_decision_engine_output[
                    "primary_action"
                ].isin(
                    actionable_actions
                ),
                "priority_tier"
            ].notna().all(),

            customer_decision_engine_output.loc[
                customer_decision_engine_output[
                    "primary_action"
                ] == "Grow",
                "recommended_product"
            ].notna().all(),

            customer_decision_engine_output.loc[
                customer_decision_engine_output[
                    "primary_action"
                ] == "Reactivate",
                "reactivation_probability"
            ].notna().all(),

            (
                customer_decision_engine_output.loc[
                    customer_decision_engine_output[
                        "customer_state"
                    ] == "Long Dormant",
                    "primary_action"
                ]
                ==
                "Unscored — Long Dormant"
            ).all(),

            repeat_snapshot_audit[
                "repeat_signal_current"
            ].sum()
            == 0,
        ],
    }
)


display(
    final_engine_checks
)


assert (
    final_engine_checks[
        "Pass"
    ].all()
)


print(
    "✓ Final Customer Decision Engine governance checks passed."
)

In [ ]:
# ================================================================
# 99.1 — EXPORT CUSTOMER DECISION ENGINE
# ================================================================

from pathlib import Path


MODEL_OUTPUT_DIR = Path(
    "model_outputs"
)

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


decision_engine_path = (
    MODEL_OUTPUT_DIR
    /
    "customer_decision_engine.parquet"
)


customer_decision_engine_output.to_parquet(
    decision_engine_path,
    index=False
)


print(
    "✓ Customer Decision Engine exported."
)

print(
    f"Path: {decision_engine_path.resolve()}"
)

print(
    f"Rows: {len(customer_decision_engine_output):,}"
)

print(
    f"Customers: "
    f"{customer_decision_engine_output['customer'].nunique():,}"
)

## Step 100 — Final Business Interpretation

The Customer Decision Engine successfully converts multiple independently developed
machine-learning signals into one governed customer-level action framework.

### 1. Customer Universe

The final decision snapshot contains:

> **5,224 unique identified customers**

Every customer appears exactly once.

The customer population is divided into four lifecycle states:

- Active
- At Risk
- Dormant
- Long Dormant

This lifecycle state determines which predictive signal is operationally relevant.

---

### 2. Final Action Architecture

The engine assigns one primary action per customer:

- **Protect** — preserve economically important active customers showing leakage exposure;
- **Retain** — intervene with customers whose purchasing recency has weakened;
- **Reactivate** — target eligible dormant customers using reactivation propensity;
- **Grow** — expand active customers using future-value and product recommendation signals;
- **Monitor** — no model-supported intervention currently takes precedence;
- **Unscored / Long Dormant** — customer lies outside current model coverage.

The resulting action population is:

| Action | Customers | Share |
|---|---:|---:|
| Protect | 458 | 8.8% |
| Retain | 1,009 | 19.3% |
| Reactivate | 509 | 9.7% |
| Grow | 284 | 5.4% |
| Monitor | 168 | 3.2% |
| Dormant — no valid Reactivation score | 340 | 6.5% |
| Long Dormant — outside model coverage | 2,456 | 47.0% |

The engine therefore avoids manufacturing recommendations for customers where
appropriate contemporaneous evidence does not exist.

---

### 3. Immediate Commercial Queue

The action-specific ranking system separates the existence of an opportunity from
the urgency of intervention.

Priority 1 and Priority 2 customers constitute the higher-priority operating queue.

The immediate queue contains:

> **680 customers**

comprising:

- **303 Retain**
- **153 Reactivate**
- **138 Protect**
- **86 Grow**

This represents approximately:

> **13.0% of the full 5,224-customer identified population**

This is materially more practical than treating every model-eligible customer as an
immediate contact requirement.

---

### 4. Contact Intensity

The decision system translates action priority into four levels of commercial effort:

- **High Touch** — Priority 1
- **Campaign** — Priority 2
- **Automated / Low Cost** — Priority 3
- **Hold / Control** — Priority 4

Customers without a valid model-led intervention receive:

> **No Model-Led Contact**

This allows human and campaign capacity to be concentrated on higher-value
opportunities rather than spreading resources uniformly across the customer base.

---

### 5. Product Recommendations

Product recommendations are only attached where the recommendation episode remains
current at the decision snapshot.

Recommendation coverage is:

- Grow: **100%**
- Protect: **77.5%**
- Retain: **25.6%**

The channel is action-aware:

- Protect / Retain → Repeat-product recommendation
- Grow → Discovery / cross-sell recommendation

Repeat recommendations carry a **GO** decision from Notebook 6.

Discovery recommendations remain **CONDITIONAL GO** and should be evaluated through
controlled commercial experiments.

---

### 6. Reactivation

Of 599 temporally available Reactivation scores, only:

> **509**

remain operationally valid for customers who are still Dormant at the decision date.

The other 90 scores are correctly suppressed because the customer's observed state
has changed.

This demonstrates why model predictions cannot be operationalised independently of
current customer state.

---

### 7. Repeat Purchase Model

All 3,561 historical Repeat Purchase scores are older than the model's 90-day
decision horizon at the August snapshot.

Therefore:

> **0 Repeat Purchase scores are used as current August customer signals.**

The Repeat model remains useful as an early-lifecycle trigger when a customer first
purchases, but it is intentionally excluded from this mature-customer snapshot.

---

### 8. Long-Dormant Customers

The largest unresolved population is:

> **2,456 Long-Dormant customers — 47.0% of identified customers**

The current model stack does not provide a contemporaneous decision signal for this
population.

The engine correctly labels them:

> **Unscored — Long Dormant**

rather than extrapolating models beyond their validated scope.

A future deep-reactivation / win-back model would be required before this population
can receive model-led prioritisation.

## Step 101 — Decision Engine Governance and Limitations

The Customer Decision Engine is an orchestration layer, not a new causal prediction
model.

Several governance limitations must remain explicit.

### 1. Historical Decision Snapshot

The current implementation reconstructs a historical decision state as of:

> **31 August 2011**

It demonstrates how the models can be integrated without temporal leakage.

Production deployment requires the upstream models to generate refreshed scores at
the live decision date.

---

### 2. Action Scores Are Ranking Scores

Protect, Retain, Reactivate and Grow priority scores are operational ranking indices.

They are **not probabilities**.

The weighting rules encode an initial business policy and have not been optimised
using observed intervention economics.

Future optimisation should incorporate:

- incremental response;
- contribution margin;
- campaign cost;
- contact-channel cost;
- customer fatigue;
- treatment capacity;
- incremental revenue or profit.

---

### 3. Propensity Is Not Causal Uplift

A customer with a high Reactivation or Retention propensity is not necessarily the
customer most influenced by an intervention.

The current models estimate behavioural likelihood or risk.

They do not estimate:

> treatment effect / incremental uplift.

Controlled treatment-control experiments are therefore required before claiming
incremental commercial impact.

---

### 4. Revenue Leakage Exposure

The Leakage exposure measure is:

> probability × 50% × baseline revenue

It is a conservative prioritisation proxy based on the leakage-event threshold.

It must not be described as a calibrated expected accounting loss.

---

### 5. Future Customer Value

The FCV signal represents expected positive merchandise revenue over the following
90 days.

It is useful primarily for relative prioritisation.

It is not:

- lifetime customer value;
- net contribution margin;
- accounting revenue forecast;
- guaranteed future revenue.

---

### 6. Product Recommendation Governance

Repeat-product recommendations have stronger validation support than Discovery
recommendations.

The Discovery channel remains:

> **CONDITIONAL GO**

and should initially be deployed through controlled experiments.

---

### 7. Inventory Constraint

The recommendation engine does not contain current stock-on-hand, stock-out,
availability or replenishment information.

A recommended product therefore represents behavioural relevance, not guaranteed
commercial availability.

A production system should introduce inventory eligibility before recommendations
are exposed to customers.

---

### 8. Long-Dormant Coverage

Customers more than 180 days since their last purchase currently fall outside the
validated customer-action model stack.

No action propensity is inferred for them.

This avoids unsupported extrapolation but leaves a material coverage gap that should
be addressed by future modelling.

---

### 9. Identified-Customer Scope

Customer-level decisions require a known customer identifier.

The engine therefore does not represent unidentified transaction revenue.

Business-level forecasting and revenue reporting must continue to account for that
revenue separately.

## Step 102 — Final Deployment Decision

# CONDITIONAL GO

The Customer Decision Engine receives a:

> **CONDITIONAL GO for monitored decision-support and controlled commercial pilots.**

### Approved Uses

The engine is suitable for:

- customer prioritisation;
- CRM work queues;
- campaign segmentation;
- retention prioritisation;
- leakage-risk prioritisation;
- controlled reactivation campaigns;
- product recommendation pilots;
- commercial resource allocation;
- treatment-control experimentation.

### Not Yet Approved For

The engine should not yet be used as:

- a fully autonomous customer-contact system;
- a causal uplift engine;
- a guaranteed revenue optimisation system;
- a calibrated profit forecast;
- an unrestricted recommendation engine without inventory checks;
- a model-led strategy for Long-Dormant customers.

### Production Promotion Requirements

Promotion beyond CONDITIONAL GO should require:

1. live recurring upstream score generation;
2. treatment-control experimentation;
3. campaign response measurement;
4. incremental-profit measurement;
5. score and population drift monitoring;
6. inventory / product-availability integration;
7. contact-frequency and customer-fatigue controls;
8. action-score weight calibration;
9. Long-Dormant strategy development;
10. production monitoring and fallback rules.

In [ ]:
# ================================================================
# 103.1 — FINAL NOTEBOOK 8 SUMMARY
# ================================================================

total_customers = len(
    customer_decision_engine_output
)

protect_customers = (
    customer_decision_engine_output[
        "primary_action"
    ]
    .eq("Protect")
    .sum()
)

retain_customers = (
    customer_decision_engine_output[
        "primary_action"
    ]
    .eq("Retain")
    .sum()
)

reactivate_customers = (
    customer_decision_engine_output[
        "primary_action"
    ]
    .eq("Reactivate")
    .sum()
)

grow_customers = (
    customer_decision_engine_output[
        "primary_action"
    ]
    .eq("Grow")
    .sum()
)

long_dormant_customers = (
    customer_decision_engine_output[
        "primary_action"
    ]
    .eq("Unscored — Long Dormant")
    .sum()
)

immediate_queue_customers = (
    customer_decision_engine_output[
        "priority_tier"
    ]
    .isin(
        [
            "Priority 1",
            "Priority 2",
        ]
    )
    .sum()
)

immediate_queue_share = (
    immediate_queue_customers
    /
    total_customers
)


notebook_8_summary = pd.DataFrame(
    {
        "Metric": [
            "Decision date",
            "Total identified customers",
            "Protect customers",
            "Retain customers",
            "Reactivate customers",
            "Grow customers",
            "Immediate Priority 1–2 queue",
            "Immediate queue share",
            "Long-Dormant / unscored customers",
            "Final deployment decision",
        ],

        "Value": [
            str(DECISION_DATE),
            f"{total_customers:,}",
            f"{protect_customers:,}",
            f"{retain_customers:,}",
            f"{reactivate_customers:,}",
            f"{grow_customers:,}",
            f"{immediate_queue_customers:,}",
            f"{immediate_queue_share:.2%}",
            f"{long_dormant_customers:,}",
            "CONDITIONAL GO",
        ],
    }
)


display(
    notebook_8_summary
)

In [ ]:
# ================================================================
# 104.1 — FINAL PERSISTENT OUTPUT AUDIT
# ================================================================

final_output_path = (
    Path("model_outputs")
    /
    "customer_decision_engine.parquet"
)


final_export_audit = pd.DataFrame(
    {
        "Check": [
            "Output file exists",
            "One row per customer",
            "Expected 5,224 rows",
            "Expected 5,224 unique customers",
            "Primary action complete",
            "Governance checks passed",
        ],

        "Pass": [
            final_output_path.exists(),

            customer_decision_engine_output[
                "customer"
            ].is_unique,

            len(
                customer_decision_engine_output
            )
            ==
            5224,

            customer_decision_engine_output[
                "customer"
            ].nunique()
            ==
            5224,

            customer_decision_engine_output[
                "primary_action"
            ].notna().all(),

            final_engine_checks[
                "Pass"
            ].all(),
        ],
    }
)


display(
    final_export_audit
)


assert (
    final_export_audit[
        "Pass"
    ].all()
)


print(
    "✓ Customer Decision Engine persistent output verified."
)

# Notebook 8 Complete — Customer Decision Engine

**Status:** COMPLETE

**Deployment Decision:** CONDITIONAL GO

The notebook has successfully:

- reconstructed a leakage-safe historical decision snapshot;
- aligned model scores according to their valid decision horizons;
- incorporated current customer lifecycle state;
- prevented stale Repeat Purchase predictions from entering the current-state engine;
- invalidated obsolete Reactivation episodes;
- invalidated stale Next-Best-Product recommendations;
- integrated Leakage, FCV, Reactivation and recommendation signals;
- assigned one governed primary action per customer;
- created action-specific priority rankings;
- translated rankings into contact-intensity tiers;
- attached product recommendations only where valid;
- created explicit experimentation requirements;
- exported a restart-safe customer decision table;
- passed final governance and uniqueness checks.

The resulting output contains:

> **5,224 unique customer decisions**

with a higher-priority intervention queue of:

> **680 customers**

The final engine is approved for:

> **monitored decision-support and controlled commercial pilots**

and is not yet approved as a fully autonomous optimisation system.